# Zone Finding — Orchestrator

Runs the Zone Finding pipeline: notebooks 01–09 extract features per census tract, then merges all CSVs into a single combined dataset for ML classification (notebook 10).

**Pipeline:**
```
01_zone_definition        → tract_id, zone_type (Y variable)
02_amenity_composition    → amenity counts, ratios, density (OSM)
03_building_characteristics → avg floors, year built, lot area (PLUTO)
04_land_use_mix           → entropy, HHI (PLUTO)
05_accessibility          → transit distances, intersection density (OSM)
06_socioeconomic          → income, population, poverty (Census)
07_tourism_intensity      → hotel count, tourism POIs (OSM)
08_pedestrian_activity    → pedestrian rank (NYC Ped Mobility)
09_commercial_density     → shop count, entropy, brand ratio (OSM)
─────────────────────────────────────────────────────────────
10_ml_classification      → train & evaluate models
```

**Usage:** Restart kernel → Run All Cells

In [1]:
import papermill as pm
import pandas as pd
import pathlib
import time
import os
import tempfile
from datetime import datetime

os.makedirs("csv", exist_ok=True)

RUN_ID = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"papermill {pm.__version__}")
print(f"Run ID: {RUN_ID}")

papermill 2.7.0
Run ID: 2026-05-14_23-48-39


In [2]:
# ── Pipeline configuration ────────────────────────────

PIPELINE = [
    ("01_zone_definition.ipynb",        "01 · Zone Definition (Y variable)"),
    ("02_amenity_composition.ipynb",     "02 · Amenity Composition (OSM)"),
    ("03_building_characteristics.ipynb", "03 · Building Characteristics (PLUTO)"),
    ("04_land_use_mix.ipynb",            "04 · Land Use Mix (PLUTO)"),
    ("05_accessibility.ipynb",           "05 · Accessibility (OSM)"),
    ("06_socioeconomic.ipynb",           "06 · Socioeconomic (Census)"),
    ("07_tourism_intensity.ipynb",       "07 · Tourism Intensity (OSM)"),
    ("08_pedestrian_activity.ipynb",     "08 · Pedestrian Activity (NYC DOT)"),
    ("09_commercial_density.ipynb",      "09 · Commercial Density (OSM)"),
]

RUN_MODULE = {nb: True for nb, _ in PIPELINE}

KERNEL_NAME = "python3"

# ML notebook runs after merge
RUN_ML = True

print("Pipeline:")
for nb, desc in PIPELINE:
    status = "RUN" if RUN_MODULE[nb] else "SKIP"
    print(f"  [{status}]  {desc}")
print(f"  [{'RUN' if RUN_ML else 'SKIP'}]  10 · ML Classification")

Pipeline:
  [RUN]  01 · Zone Definition (Y variable)
  [RUN]  02 · Amenity Composition (OSM)
  [RUN]  03 · Building Characteristics (PLUTO)
  [RUN]  04 · Land Use Mix (PLUTO)
  [RUN]  05 · Accessibility (OSM)
  [RUN]  06 · Socioeconomic (Census)
  [RUN]  07 · Tourism Intensity (OSM)
  [RUN]  08 · Pedestrian Activity (NYC DOT)
  [RUN]  09 · Commercial Density (OSM)
  [RUN]  10 · ML Classification


In [ ]:
# ── Run feature extraction notebooks ──────────────────

results = []

for nb_path, description in PIPELINE:
    if not RUN_MODULE.get(nb_path, True):
        print(f"  SKIP  {description}")
        results.append((nb_path, "skipped", 0))
        continue
    
    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")
    
    print(f"\n{'='*55}")
    print(f"  {description}")
    print(f"{'='*55}")
    
    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path, str(tmp),
            kernel_name=KERNEL_NAME,
            parameters={"ZONES_CONFIG": "zones.json"},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
        results.append((nb_path, "ok", elapsed))
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
        results.append((nb_path, "failed", elapsed))
    finally:
        if tmp.exists():
            tmp.unlink()

# ── Summary ───────────────────────────────────────────
print(f"\n{'#'*55}")
print(f"  PIPELINE SUMMARY")
print(f"{'#'*55}")
for nb_path, status, elapsed in results:
    icon = {"ok": "OK", "failed": "FAIL", "skipped": "SKIP"}.get(status, "?")
    t_str = f"{elapsed:.1f} s" if elapsed else "-"
    print(f"  [{icon:4s}]  {nb_path:<42s} {t_str}")

failed = [nb for nb, s, _ in results if s == "failed"]
if failed:
    print(f"\n  WARNING: {len(failed)} notebook(s) failed: {failed}")


  01 · Zone Definition (Y variable)
  Status : OK  (4.5 s)

  02 · Amenity Composition (OSM)
  Status : OK  (1461.9 s)

  03 · Building Characteristics (PLUTO)
  Status : OK  (4.3 s)

  04 · Land Use Mix (PLUTO)
  Status : OK  (4.7 s)

  05 · Accessibility (OSM)


In [ ]:
# ── Merge all CSV outputs ─────────────────────────────

CSV_GROUPS = [
    ("csv/01_zone_definition.csv", ["tract_id", "borough", "tract_lat", "tract_lon",
                                     "zone_type", "tract_lot_count"]),
    ("csv/02_amenity_composition.csv", None),  # all columns
    ("csv/03_building_characteristics.csv", None),
    ("csv/04_land_use_mix.csv", None),
    ("csv/05_accessibility.csv", None),
    ("csv/06_socioeconomic.csv", None),
    ("csv/07_tourism_intensity.csv", None),
    ("csv/08_pedestrian_activity.csv", None),
    ("csv/09_commercial_density.csv", None),
]

# Start with zone definitions
df_combined = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Base: {len(df_combined)} tracts from 01_zone_definition")

# Merge each subsequent CSV on tract_id
for csv_path, cols in CSV_GROUPS[1:]:
    if pathlib.Path(csv_path).exists():
        df_other = pd.read_csv(csv_path, dtype={"tract_id": str})
        # Select specific columns if specified, otherwise all except tract_id (already in base)
        merge_cols = [c for c in df_other.columns if c != "tract_id"]
        df_combined = df_combined.merge(
            df_other[["tract_id"] + merge_cols],
            on="tract_id", how="left"
        )
        print(f"  OK    {csv_path} (+{len(merge_cols)} cols)")
    else:
        print(f"  WARN  {csv_path} not found — columns will be NaN")

print(f"\nCombined shape: {df_combined.shape}")

In [ ]:
# ── Save combined CSV ─────────────────────────────────

combined_path = f"csv/combined_zones_{RUN_ID}.csv"
df_combined.to_csv(combined_path, index=False, encoding="utf-8")
print(f"Saved: {combined_path}")
print(f"  {df_combined.shape[0]} tracts x {df_combined.shape[1]} columns")

# Column completeness
print(f"\nColumn completeness:")
for col in df_combined.columns:
    n = df_combined[col].notna().sum()
    pct = 100 * n / len(df_combined)
    print(f"  {col:<35s} {n:>4d}/{len(df_combined)}  ({pct:.1f}%)")

In [ ]:
# ── Preview ───────────────────────────────────────────
print(df_combined.dtypes)
print()
df_combined.head(10)

In [ ]:
# ── Run ML notebook ──────────────────────────────────

if RUN_ML:
    print(f"\n{'='*55}")
    print(f"  10 · ML Classification")
    print(f"{'='*55}")
    
    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            "10_ml_classification.ipynb", str(tmp),
            kernel_name=KERNEL_NAME,
            parameters={"CSV_PATH": combined_path},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
    finally:
        if tmp.exists():
            tmp.unlink()
else:
    print("ML notebook skipped (RUN_ML = False)")

print(f"\n{'#'*55}")
print(f"  ZONE FINDING PIPELINE COMPLETE")
print(f"  Combined CSV: {combined_path}")
print(f"{'#'*55}")